In [1]:
!bash --login -c "poetry update"

Updating dependencies
Resolving dependencies... (3.1s)Resolving dependencies... (1.6s)

No dependencies to install or update


# Option valuation model based on historical prices

This code assumes `aws configure` has been run in the environment of the notebook.

In [4]:
import optionspricing
import boto3
import helpers
import pandas
import dotenv

dotenv.load_dotenv()

BINANCE_ETH = "ETHUSDT"
BINANCE_BTC_USDT = "BTCUSDT"
BINANCE_BTC = "BTCUSDC"
BINANCE_XRP = "XRPUSDC"
BINANCE_SOL = "SOLUSDC"
BINANCE_BNB = "BNBUSDC"

BINANCE_SYMBOL = BINANCE_ETH
  
INVERSE_QUOTES = ["ETHUSDT", "BTCUSDT"]
quote_in_usd = BINANCE_SYMBOL not in INVERSE_QUOTES

s3 = boto3.resource('s3')

data_file = helpers.fetch_object(s3, "test-binance-prices-255120844515", f"{BINANCE_SYMBOL}-full.csv.zip")
prices_df = pandas.read_csv(data_file, compression='zip', header=0, index_col="dateTime") # pyright: ignore[reportArgumentType]
prices_df.index = pandas.to_datetime(prices_df.index)

## Option valuation model: input parameters here

In [6]:
target_period_hours = 3 * 24
strikes_universe_size = 4
CUT_OFF_YEAR_MONTH = (2021, 7)

instrument_code = BINANCE_SYMBOL

headers = {"Content-Type": "application/json"}
base_url = "https://www.deribit.com/api/v2/public"

trading_model = optionspricing.TradingModel(base_url, headers, instrument_code, target_period_hours)
trading_model.cutoff_year_month(CUT_OFF_YEAR_MONTH)

option_chain_df = trading_model.evaluate(prices_df, strikes_universe_size)
simulation = trading_model.simulate_strategy_long_straddle(option_chain_df, strikes_universe_size, quote_in_usd=quote_in_usd) # pyright: ignore[reportArgumentType]
print(simulation)

loaded 436 puts and 436 calls for ETHUSDT
target expiry: Sun 12 Oct, 08:00 (72 hours left)
current price: 4436.19
-------------------------------
hit ratio: 45%
cost: 0.033 / value: 0.041, gain% = 0.87%
($) cost: 144.18 / value: 182.70, average gain = 38.53
buy put 4400.0
buy call 4450.0
-------------------------------
hit ratio: 44%
cost: 0.028 / value: 0.036, gain% = 0.84%
($) cost: 124.21 / value: 161.59, average gain = 37.37
buy put 4375.0
buy call 4475.0
-------------------------------
hit ratio: 43%
cost: 0.024 / value: 0.032, gain% = 0.87%
($) cost: 104.25 / value: 142.72, average gain = 38.47
buy put 4350.0
buy call 4500.0
-------------------------------
hit ratio: 40%
cost: 0.018 / value: 0.027, gain% = 0.84%
($) cost: 82.07 / value: 119.47, average gain = 37.40
buy put 4300.0
buy call 4525.0


# Data for covered calls

In [10]:
call_data = option_chain_df[["value_call", "value_call_pct", "call_bid"]].copy()
call_data["call_bid_usd"] = option_chain_df["value_call"].mul(option_chain_df["call_bid"].div(option_chain_df["value_call_pct"]))
call_data[["value_call", "call_bid_usd"]]

,value_call,call_bid_usd
strike,,
2725.0,80.897423,NaN
2750.0,61.610356,NaN
2775.0,44.777260,NaN
2800.0,31.601750,40.465585
2825.0,22.392156,30.698030
2850.0,15.955425,23.721205
2875.0,11.412055,19.535110


# Sanity checks

In [14]:

trading_model = optionspricing.TradingModel(base_url, headers, instrument_code, target_period_hours)
size = 6
option_chain_df = trading_model.evaluate(prices_df, strikes_universe_size=size)
put_weights = [0.] * (2 * size + 1)
call_weights = [0.] * (2 * size + 1)
put_weights[size - 5] = +1.
put_weights[size - 1] = -1.
call_weights[size + 1] = -1.
call_weights[size + 5] = +1.

simulation = trading_model.simulate_strategy(option_chain_df, put_weights, call_weights, quote_in_usd=quote_in_usd)
print(simulation)

target expiry: Sun 07 Jul, 08:00 (49 hours left)
current price: 2858.15
-------------------------------
hit ratio: 58%
cost: -0.021 / value: -0.020, gain% = 0.15%
($) cost: -61.45 / value: -57.16, average gain = 4.29
buy put 2600.0
sell put 2800.0
sell call 2900.0
buy call 3000.0


In [7]:
prices_df

,open,high,low,close,volume
dateTime,,,,,
2018-12-15 03:00:00,3200.00,3312.32,3000.00,3225.97,2.374006
2018-12-15 04:00:00,3225.97,3228.10,3205.58,3228.10,2.410518
2018-12-15 05:00:00,3228.10,3228.10,3204.06,3222.87,3.514068
2018-12-15 06:00:00,3225.68,3225.80,3199.87,3199.87,2.220411
2018-12-15 07:00:00,3199.88,3220.15,3191.44,3205.42,46.164846
...,...,...,...,...,...
2025-08-31 19:00:00,109162.00,109182.00,108923.90,108944.00,49.137060
2025-08-31 20:00:00,108944.00,109140.00,108885.11,109139.99,53.103290
2025-08-31 21:00:00,109139.99,109140.00,108961.23,109054.20,44.413170
